## Speech Pipeline — Whisper STT + emotion2vec+ Acoustic Emotion
**Project:** NLP-Based Emotion-Aware Journal Analysis System  
**Pipeline:** Voice Entry → Whisper STT → XLM-RoBERTa (linguistic) + emotion2vec+ (acoustic)  
**Both pipelines are fully independent — text and speech never merge here**

### 1. Install Dependencies

In [20]:
# !pip install librosa soundfile numpy pandas torch

In [21]:
import os
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

### Downloading Model

In [22]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Path to dataset files: /kaggle/input/ravdess-emotional-speech-audio


### 2. Device Setup

In [24]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS")
else:
    DEVICE = "cpu"
    print("CPU")


print(f"Device: {DEVICE}")

CUDA: Tesla T4
Device: cuda


### 3.Data Preprocessing

In [23]:
def parse_ravdess(path):
    data = []

    for actor in os.listdir(path):
        actor_path = os.path.join(path, actor)

        if not os.path.isdir(actor_path):
            continue

        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                parts = file.split("-")
                emotion = int(parts[2])  # emotion code

                full_path = os.path.join(actor_path, file)
                data.append((full_path, emotion - 1))  # zero index

    return pd.DataFrame(data, columns=["path", "emotion"])

In [25]:
DATA_PATH = "/kaggle/input/ravdess-emotional-speech-audio/audio_speech_actors_01-24"

df = parse_ravdess(DATA_PATH)

print("Total samples:", len(df))
df.head()

Total samples: 1440


,path,emotion
0,/kaggle/input/ravdess-emotional-speech-audio/a...,7
1,/kaggle/input/ravdess-emotional-speech-audio/a...,0
2,/kaggle/input/ravdess-emotional-speech-audio/a...,6
3,/kaggle/input/ravdess-emotional-speech-audio/a...,6
4,/kaggle/input/ravdess-emotional-speech-audio/a...,0


In [36]:
emotion_map = {
    0: 0,  # neutral
    1: 0,  # calm → neutral
    2: 1,  # happy
    3: 2,  # sad
    4: 3,  # angry
    5: 4,  # fear
    6: 5,  # disgust
    7: 6   # surprise
}

In [37]:
def extract_features(file_path, max_len=200):
    y, sr = librosa.load(file_path, sr=22050)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.vstack([mfcc, delta, delta2])  # 120 features

    # Normalize
    features = (features - np.mean(features)) / (np.std(features) + 1e-6)

    if features.shape[1] < max_len:
        pad = max_len - features.shape[1]
        features = np.pad(features, ((0,0),(0,pad)))
    else:
        features = features[:, :max_len]

    return features

### 4.Train Test Split

In [26]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["emotion"])

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["emotion"])

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 1152
Val: 144
Test: 144


### 5. Dataset Class

In [42]:
class SpeechDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 🔥 NEW FEATURE FUNCTION
        features = extract_features(row["path"])   # (120, 200)

        features = torch.tensor(features, dtype=torch.float)

        # 🔥 add channel dimension → (1, 120, 200)
        features = features.unsqueeze(0)

        label = torch.tensor(row["emotion"], dtype=torch.long)

        return features, label

In [43]:
train_loader = DataLoader(SpeechDataset(train_df), batch_size=16, shuffle=True)
val_loader = DataLoader(SpeechDataset(val_df), batch_size=16)
test_loader = DataLoader(SpeechDataset(test_df), batch_size=16)

### 6. Model Class

In [44]:
class SpeechEmotionModel(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 30 * 50, 128),  # works for MFCC (40x200)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SpeechEmotionModel().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

In [46]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

### 7. Training

In [47]:
best_val_acc = 0

for epoch in range(20):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {total_loss/len(train_loader):.4f}")
    print(f"Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_speech_model.pth")
        print("Best model saved!")

Epoch 1
Train Loss: 2.0416
Val Acc: 0.3125
Best model saved!
Epoch 2
Train Loss: 1.8141
Val Acc: 0.3889
Best model saved!
Epoch 3
Train Loss: 1.6532
Val Acc: 0.4028
Best model saved!
Epoch 4
Train Loss: 1.5787
Val Acc: 0.4861
Best model saved!
Epoch 5
Train Loss: 1.4439
Val Acc: 0.4306
Epoch 6
Train Loss: 1.3774
Val Acc: 0.4792
Epoch 7
Train Loss: 1.2747
Val Acc: 0.5000
Best model saved!
Epoch 8
Train Loss: 1.2273
Val Acc: 0.5278
Best model saved!
Epoch 9
Train Loss: 1.1205
Val Acc: 0.5208
Epoch 10
Train Loss: 1.0448
Val Acc: 0.5903
Best model saved!
Epoch 11
Train Loss: 0.9540
Val Acc: 0.5000
Epoch 12
Train Loss: 0.9160
Val Acc: 0.5764
Epoch 13
Train Loss: 0.8596
Val Acc: 0.5972
Best model saved!
Epoch 14
Train Loss: 0.8131
Val Acc: 0.5903
Epoch 15
Train Loss: 0.7697
Val Acc: 0.6111
Best model saved!
Epoch 16
Train Loss: 0.7094
Val Acc: 0.6111
Epoch 17
Train Loss: 0.6432
Val Acc: 0.5903
Epoch 18
Train Loss: 0.6124
Val Acc: 0.6250
Best model saved!
Epoch 19
Train Loss: 0.5695
Val Acc: 

In [48]:
model.load_state_dict(torch.load("best_speech_model.pth"))

test_acc = evaluate(model, test_loader)

print("Test Accuracy:", test_acc)

Test Accuracy: 0.6388888888888888
